# Code Auditor - AST Parsing & CodeBERT Representations

This notebook demonstrates how we audit a candidate's uploaded codebase using Python's built-in Abstract Syntax Tree (`ast`) parser to extract structure and components, and how we can use a pre-trained `CodeBERT` model (`microsoft/codebert-base`) to obtain semantic code embeddings.

In [1]:
import os
import ast
import torch
from transformers import AutoTokenizer, AutoModel

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Abstract Syntax Tree (AST) Parsing

We read a local code file, parse it into an AST, and extract defined function names, imports, and classes without executing the code.

In [2]:
code_snippet = """
import os
from fastapi import FastAPI

app = FastAPI()

def calculate_salary(base, experience):
    if experience > 5:
        return base * 1.2
    return base
"""

tree = ast.parse(code_snippet)

functions = []
imports = []

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        functions.append(node.name)
    elif isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
        imports.append(ast.dump(node))

print("Extracted Function Definitions:", functions)
print("Extracted Import Definitions:", len(imports))

Extracted Function Definitions: ['calculate_salary']
Extracted Import Definitions: 2


## 2. Load Pre-trained CodeBERT Model

We load Microsoft's `codebert-base` model. It represents code tokens in a high-dimensional vector space, capturing programming logic semantics.

In [3]:
model_name = "microsoft/codebert-base"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    # Generate embeddings for a function block
    code_tokens = tokenizer.tokenize("def calculate_salary(base, experience):")
    tokens_ids = tokenizer.convert_tokens_to_ids(code_tokens)
    
    # Convert to tensor and run forward pass
    input_tensor = torch.tensor([tokens_ids])
    with torch.no_grad():
        outputs = model(input_tensor)[0]
        
    print(f"Generated representation for code snippet. Vector shape: {outputs.shape}")
except Exception as e:
    print(f"Skipping CodeBERT load during demo. Details: {e}")

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vansh Agrawal\.cache\huggingface\hub\models--microsoft--codebert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Generated representation for code snippet. Vector shape: torch.Size([1, 10, 768])
